# Chapter 6 — Training optimization and DPO
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch06_training_optimization_dpo.ipynb)

Covers AdamW, warmup/cosine scheduling, BF16 mixed precision, gradient clipping, validation, and the DPO objective. The notebook uses a small model so every training cell is practical on a T4.

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F
device='cuda' if torch.cuda.is_available() else 'cpu'
print(device, torch.cuda.get_device_name(0) if device=='cuda' else '')

## 1. AdamW and warmup + cosine decay

In [ ]:
def lr_at(step,warmup,total,max_lr=3e-4,min_lr=3e-5):
    if step < warmup: return max_lr*(step+1)/warmup
    r=(step-warmup)/max(1,total-warmup)
    c=0.5*(1+math.cos(math.pi*min(r,1.0)))
    return min_lr+c*(max_lr-min_lr)
print([lr_at(s,10,100) for s in [0,9,10,50,99]])

## 2. Tiny LM + BF16 training loop

In [ ]:
class LM(nn.Module):
    def __init__(self,vocab=256,d=128):
        super().__init__(); self.emb=nn.Embedding(vocab,d); self.rnn=nn.GRU(d,d,batch_first=True); self.head=nn.Linear(d,vocab)
    def forward(self,x):
        h,_=self.rnn(self.emb(x)); return self.head(h)
model=LM().to(device); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=0.1)
for step in range(40):
    x=torch.randint(0,256,(32,96),device=device); y=x.roll(-1,1)
    for g in opt.param_groups: g['lr']=lr_at(step,5,40)
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda',dtype=torch.bfloat16,enabled=device=='cuda'):
        loss=F.cross_entropy(model(x).reshape(-1,256),y.reshape(-1))
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
    if step%10==0: print(step,float(loss),opt.param_groups[0]['lr'])

## 3. Validation loop

In [ ]:
@torch.no_grad()
def validate(model,batches=10):
    model.eval(); vals=[]
    for _ in range(batches):
        x=torch.randint(0,256,(16,96),device=device); y=x.roll(-1,1)
        vals.append(F.cross_entropy(model(x).reshape(-1,256),y.reshape(-1)).item())
    model.train(); return sum(vals)/len(vals)
print('val loss=',validate(model))

## 4. DPO loss from scratch
For preference pair `(chosen, rejected)`, DPO compares the policy log-probability gap against a frozen reference-model gap.

In [ ]:
def dpo_loss(pi_chosen,pi_rejected,ref_chosen,ref_rejected,beta=0.1):
    pi_gap=pi_chosen-pi_rejected; ref_gap=ref_chosen-ref_rejected
    return -F.logsigmoid(beta*(pi_gap-ref_gap)).mean()
pi_c=torch.tensor([-2.0,-1.5],requires_grad=True); pi_r=torch.tensor([-2.4,-2.2],requires_grad=True)
ref_c=torch.tensor([-2.1,-1.7]); ref_r=torch.tensor([-2.3,-2.0])
loss=dpo_loss(pi_c,pi_r,ref_c,ref_r); loss.backward()
print('DPO loss=',float(loss),'grad chosen=',pi_c.grad.tolist())

## T4 note
The full chapter's StoryBot training can be longer, but AdamW, scheduler, BF16, clipping, validation and DPO all fit comfortably on T4. This notebook prioritizes algorithm verification over long training. LLM-as-a-Judge requires an external judge model/API, so it is described but not made a required execution step.

Upstream: https://github.com/oreilly-japan/deep-learning-from-scratch-6/tree/main/ch06